In [0]:

%pip install sentence-transformers transformers faiss-cpu
dbutils.library.restartPython()


In [0]:
def chunk_text(text, chunk_size=50):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]





text = "Databricks supports Apache Spark for large scale analytics"
chunks = chunk_text(text, chunk_size=3)
print(chunks)


[
  "Databricks supports Apache",
  "Spark for large",
  "scale analytics"
]

*/

In [0]:
documents = [
    "Databricks supports Apache Spark for large-scale data processing.",
    "Delta Lake provides ACID transactions on data lakes.",
    "MLflow is used for experiment tracking and model lifecycle management."
]

In [0]:
chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

In [0]:
display(chunks)

In [0]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)


In [0]:
import pandas as pd

df = pd.DataFrame({
    "text": chunks,
    "embedding": embeddings.tolist()
})

spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable("genai_embeddings")

In [0]:
%sql

select * from genai_embeddings

In [0]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def search_similar(query, top_k=2):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [chunks[i] for i in top_indices]

In [0]:
def build_prompt(context, question):
    return f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

In [0]:
from transformers import pipeline

qa_pipeline = pipeline("text-generation", model="google/flan-t5-small")

context = "\n".join(search_similar("What is Delta Lake?"))
prompt = build_prompt(context, "What is Delta Lake?")
qa_pipeline(prompt, max_length=200)

In [0]:
from transformers import pipeline

qa_pipeline = pipeline("text2text-generation",model="google/flan-t5-small")

context = "\n".join(search_similar("What is Delta Lake?"))
prompt = build_prompt(context, "What is Delta Lake?")
result = qa_pipeline(prompt, max_new_tokens=100)
answer = result[0]["generated_text"]

print(answer)



In [0]:
import mlflow

with mlflow.start_run():
    mlflow.log_param("embedding_model", "MiniLM")
    mlflow.log_metric("num_chunks", len(chunks))